# Local Agentic AI Workshop Notebook (Updated)
This notebook works with your latest `agent.py` and `tools.py` (2-agent Planner + Executor + timings). It demonstrates:

- Local Ollama connectivity
- Memory updates
- Planner output (natural language + JSON)
- Tool execution
- Executor output
- Timing breakdown

**Note:** This notebook does *not* require any cloud provider.


## 0) Prerequisites (one-time)

1) Create/activate a virtual environment (recommended).
2) Install dependencies:

```bash
pip install -r requirements.txt
```

3) Make sure Ollama is installed and running:

```bash
ollama serve
ollama pull llama3.2:1b
```

If you use a different model, update the `MODEL` variable below.


In [1]:
# 1) Basic configuration
MODEL = "llama3.2:1b"   # change if needed
TEMPERATURE = 0.2
MEMORY_PATH = "memory.json"


In [2]:
# 2) Quick health check: is Ollama responding?
import requests, json, time

def ollama_tags(base_url="http://localhost:11434"):
    r = requests.get(f"{base_url}/api/tags", timeout=10)
    r.raise_for_status()
    return r.json()

tags = ollama_tags()
print("Ollama OK. Models available:")
for m in tags.get("models", []):
    print(" -", m.get("name"))


Ollama OK. Models available:
 - llama3.2:1b


## 1) Import the agent code
This uses your local `agent.py` and `tools.py`.


In [3]:
from agent import (
    load_memory, save_memory, maybe_update_memory,
    agent_reply, agent_reply_with_full_trace
)

memory = load_memory(MEMORY_PATH)
print("Loaded memory profile:", memory.get("profile"))


Loaded memory profile: {'name': 'Vinay', 'preferences': {'pref_1': 'prefer vegetarian meals'}}


## 2) Memory demo
Run the next cell to store name and a preference.


In [4]:
memory = load_memory(MEMORY_PATH)

print("Before:", memory.get("profile"))

maybe_update_memory("My name is Vinay", memory)
maybe_update_memory("Remember that I prefer vegetarian meals", memory)

save_memory(memory, MEMORY_PATH)

memory2 = load_memory(MEMORY_PATH)
print("After:", memory2.get("profile"))


Before: {'name': 'Vinay', 'preferences': {'pref_1': 'prefer vegetarian meals'}}
After: {'name': 'Vinay', 'preferences': {'pref_1': 'prefer vegetarian meals', 'pref_2': 'prefer vegetarian meals'}}


## 3) Single-agent-style call (final answer only)
This calls the full 2-agent pipeline, but returns only the final answer.


In [5]:
prompt = "Create a checklist for hosting a kids birthday party for 12 people."
answer = agent_reply(prompt, memory_path=MEMORY_PATH, model=MODEL, temperature=TEMPERATURE)
print(answer)
print("\n---\nAI Generated Content please check it")


Here's a simple checklist for hosting a kids birthday party for 12 people:

**Pre-Party Planning**

* Send out invitations
* Plan the menu (e.g., vegetarian options)
* Book a venue
* Create a guest list
* Send out reminders
* Prepare party decorations

**Food and Drinks**

* Plan the menu (e.g., vegetarian options)
* Order food or make it yourself
* Prepare party snacks (e.g., cupcakes, fruit)

**Decorations and Party Supplies**

* Plan the decorations (e.g., balloons, streamers)
* Buy party supplies (e.g., plates, cups, napkins)
* Set up tables and chairs

**Logistics**

* Confirm the venue and catering details
* Plan for parking and transportation (if needed)
* Prepare a first aid kit

**Miscellaneous**

* Create a playlist or find a kid-friendly music source
* Prepare party favors (e.g., stickers, coloring books)

This checklist should help you stay organized and ensure a fun and memorable birthday party for the kids!

---
AI Generated Content please check it


## 4) Full trace: Planner + Tool + Executor + Timings
This is the best cell for workshop demos.


In [6]:
from pprint import pprint

prompt = """I’m hosting a kids birthday party for 12 people this weekend.
Create a simple plan and also generate a checklist."""

trace = agent_reply_with_full_trace(prompt, memory_path=MEMORY_PATH, model=MODEL, temperature=TEMPERATURE)

print("\n=== Planner (Natural Language) ===")
print(trace["planner_output"].get("planner_message",""))

print("\n=== Planner (Structured JSON) ===")
pprint(trace["planner_output"])

print("\n=== Tool Output ===")
print(trace["tool_result"] if trace["tool_result"] else "(No tool used)")

print("\n=== Executor (Final Answer) ===")
print(trace["executor_answer"])
print("\n---\nAI Generated Content please check it")

print("\n=== Timings (seconds) ===")
pprint(trace.get("timings", {}))



=== Planner (Natural Language) ===
I will respond directly without a tool.

=== Planner (Structured JSON) ===
{'_raw': '{\n'
         '  "planner_message": "Plan a fun kids\' birthday party for 12 people '
         'this weekend.",\n'
         '  "plan": ["Invite guests", "Choose a venue", "Plan games", "Prepare '
         'food", "Decorate the party area"],\n'
         '  "clarifying_questions": [],\n'
         '  "tool": null,\n'
         '  "args": {}\n'
         '}\n'
         '\n'
         '{\n'
         '  "planner_message": "Create a simple checklist for the party.",\n'
         '  "plan": ["Invitation sent", "Venue chosen", "Games planned", "Food '
         'ordered", "Decorations done"],\n'
         '  "clarifying_questions": [],\n'
         '  "tool": null,\n'
         '  "args": {}\n'
         '}',
 'args': {},
 'clarifying_questions': [],
 'plan': [],
 'planner_message': 'I will respond directly without a tool.',
 'tool': None}

=== Tool Output ===
(No tool used)

=== Exec

## 5) Try more prompts
These are good for showing different tools and behaviors.


In [7]:
test_prompts = [
    "I have 4 hours tonight. Create a focused time-block plan to study for a cybersecurity midterm.",
    "Help me decide whether I should buy an e-bike this month. My budget is $600.",
    "Plan a fun Sunday for me under $40 and give me a checklist."
]

for p in test_prompts:
    print("\n" + "="*80)
    print("PROMPT:", p)
    t = agent_reply_with_full_trace(p, memory_path=MEMORY_PATH, model=MODEL, temperature=TEMPERATURE)
    print("\nPlanner:", t["planner_output"].get("planner_message",""))
    print("\nExecutor:", t["executor_answer"])
    print("\n---\nAI Generated Content please check it")
    print("\nTimings:", t.get("timings", {}))



PROMPT: I have 4 hours tonight. Create a focused time-block plan to study for a cybersecurity midterm.

Planner: Create a focused 4-hour time-block plan to study for a cybersecurity midterm.

Executor: I'd be happy to help you create a focused time-block plan to study for your cybersecurity midterm.

Considering your 4-hour time block, here's a suggested plan:

**Time Block 1: Review Notes and Study Materials (9:00 AM - 10:30 AM)**

* Start by reviewing your notes and study materials for the midterm. Focus on key concepts and topics that you haven't covered yet.
* Take notes as you go, and review them later to reinforce your understanding.

**Time Block 2: Practice Past Exam Questions (10:30 AM - 12:00 PM)**

* Move on to practicing past exam questions or case studies. This will help you assess your knowledge and identify areas where you need more practice.
* Try to answer as many questions as possible within the given time frame.

**Time Block 3: Focus on Weak Areas (12:00 PM - 1:30 

## 6) Troubleshooting quick fixes

- If you see timeouts, try a smaller model or reduce temperature.
- If Streamlit works but notebook fails, restart the notebook kernel so it reloads `agent.py`.
- If the planner returns non-JSON, the code falls back to a safe default and the executor will still respond.
